In [ ]:
from __future__ import annotations
from collections.abc import Callable
from egglog import *

array_ruleset = ruleset(name="array_ruleset")


class Boolean(Expr):
    def __init__(self, val: BoolLike) -> None: ...
    def if_bool(self, then: Int, else_: Int) -> Int: ...


class Int(Expr):
    @classmethod
    def var(cls, name: StringLike) -> Int: ...

    def __init__(self, val: i64Like) -> None: ...
    def __eq__(self, other: Int) -> Boolean: ...  # type: ignore[override]
    def __lt__(self, other: Int) -> Boolean: ...
    def __add__(self, other: Int) -> Int: ...
    def __sub__(self, other: Int) -> Int: ...


@array_ruleset.register
def _int(i: i64, j: i64, x: Int, y: Int):
    yield rewrite(Int(i) + Int(j)).to(Int(i + j))
    yield rewrite(Int(i) - Int(j)).to(Int(i - j))
    yield rewrite(Int(i) == Int(i)).to(Boolean(True))
    yield rewrite(Int(i) == Int(j)).to(Boolean(False), i != j)
    yield rewrite(Int(i) < Int(j)).to(Boolean(True), i < j)
    yield rewrite(Int(i) < Int(j)).to(Boolean(False), i >= j)
    yield rewrite(Boolean(True).if_bool(x, y)).to(x)
    yield rewrite(Boolean(False).if_bool(x, y)).to(y)


@function
def vec_index(vec: Vec[Int], index: Int) -> Int: ...


@array_ruleset.register
def _vec_index(i: i64, xs: Vec[Int]):
    yield rewrite(vec_index(xs, Int(i))).to(xs[i])


class TupleInt(Expr, ruleset=array_ruleset):
    def __init__(self, length: Int, getitem_fn: Callable[[Int], Int]) -> None: ...
    def __getitem__(self, index: Int) -> Int: ...

    @property
    def length(self) -> Int: ...

    @classmethod
    def from_vec(cls, xs: Vec[Int]) -> TupleInt:
        return TupleInt(
            Int(xs.length()),
            lambda i: vec_index(xs, i),
        )


@array_ruleset.register
def _tuple_int(l: Int, fn: Callable[[Int], Int], i: Int):
    ti = TupleInt(l, fn)
    yield rewrite(ti.length).to(l)
    yield rewrite(ti[i]).to(fn(i))


class NDArray(Expr, ruleset=array_ruleset):
    def __init__(self, shape: TupleInt, idx_fn: Callable[[TupleInt], Int]) -> None: ...

    @classmethod
    def from_vec(cls, values: Vec[Int]) -> NDArray:
        return NDArray(
            TupleInt(Int(1), lambda i: Int(values.length())),
            lambda idx: vec_index(values, idx[Int(0)]),
        )

    def with_shape(self, shape: TupleInt) -> NDArray:
        return NDArray(shape, self.__getitem__)

    @classmethod
    def var(cls, name: StringLike) -> NDArray: ...

    @property
    def shape(self) -> TupleInt: ...

    def __getitem__(self, index: TupleInt) -> Int: ...


@array_ruleset.register
def _ndarray(shape: TupleInt, fn: Callable[[TupleInt], Int], idx: TupleInt):
    nda = NDArray(shape, fn)
    yield rewrite(nda.shape).to(shape)
    yield rewrite(nda[idx]).to(fn(idx))


@function(subsume=True, ruleset=array_ruleset)
def cat(l: NDArray, r: NDArray) -> NDArray:
    """
    Returns the concatenation of two arrays, they should have the same shape and the first dimension is added.
    """
    return NDArray(
        TupleInt(
            l.shape.length,
            lambda i: (i == Int(0)).if_bool(l.shape[Int(0)] + r.shape[Int(0)], l.shape[i]),
        ),
        lambda idx: (idx[Int(0)] < l.shape[Int(0)]).if_bool(
            l[idx], r[TupleInt(r.shape.length, lambda i: (i == Int(0)).if_bool(idx[Int(0)] - l.shape[Int(0)], idx[i]))]
        ),
    )


@function(subsume=True, ruleset=array_ruleset)
def drop(x: Int, arr: NDArray) -> NDArray:
    """
    Drops the first `x` elements off the front of the array `arr`.
    """
    return NDArray(
        TupleInt(
            arr.shape.length,
            lambda i: (i == Int(0)).if_bool(arr.shape[Int(0)] - x, arr.shape[i]),
        ),
        lambda idx: arr[
            TupleInt(
                arr.shape.length,
                #  Add x to the first index, so it skips the first x elements
                lambda i: (i == Int(0)).if_bool(idx[Int(0)] + x, idx[i]),
            )
        ],
    )


@function(subsume=True, ruleset=array_ruleset)
def take(x: Int, arr: NDArray) -> NDArray:
    """
    Takes the first `x` elements off the front of the array `arr`.
    """
    return NDArray(
        TupleInt(
            arr.shape.length,
            lambda i: (i == Int(0)).if_bool(x, arr.shape[i]),
        ),
        lambda idx: arr[idx],
    )

In [ ]:
shape = TupleInt.from_vec(Vec(Int(2), Int(3), Int(4)))
RAMY = NDArray.var("RAMY").with_shape(shape)
AMY = NDArray.var("AMY").with_shape(shape)


egraph = EGraph()

Amts = egraph.let("Amts", take(Int(2), drop(Int(2), cat(RAMY, AMY))))

ndim = egraph.let("ndim", Amts.shape.length)
shape_1 = egraph.let("shape_1", Amts.shape[Int(0)])
shape_2 = egraph.let("shape_2", Amts.shape[Int(1)])
shape_3 = egraph.let("shape_3", Amts.shape[Int(2)])
idxed = egraph.let("idxed", Amts[TupleInt.from_vec(Vec(Int.var("i"), Int.var("j"), Int.var("k")))])

egraph.run(array_ruleset.saturate())
print(f"Amts.shape.length()={egraph.extract(ndim)}")
print(f"Amts.shape[0]={egraph.extract(shape_1)}")
print(f"Amts.shape[1]={egraph.extract(shape_2)}")
print(f"Amts.shape[2]={egraph.extract(shape_3)}")
print(f"Amts[i, j, k]=\n{egraph.extract(idxed)}")

Amts.shape.length()=Int(3)
Amts.shape[0]=Int(2)
Amts.shape[1]=Int(3)
Amts.shape[2]=Int(4)
Amts[i, j, k]=
_TupleInt_1 = TupleInt(
    Int(3),
    lambda i: (i == Int(0)).if_bool(
        TupleInt.from_vec(Vec[Int](Int.var("i"), Int.var("j"), Int.var("k")))[Int(0)] + Int(2), TupleInt.from_vec(Vec[Int](Int.var("i"), Int.var("j"), Int.var("k")))[i]
    ),
)
((Int.var("i") + Int(2)) < Int(2)).if_bool(
    NDArray.var("RAMY")[_TupleInt_1],
    NDArray.var("AMY")[
        TupleInt(
            Int(3),
            lambda i: (i == Int(0)).if_bool(
                _TupleInt_1[Int(0)] - NDArray.var("RAMY").with_shape(TupleInt.from_vec(Vec[Int](Int(2), Int(3), Int(4)))).shape[Int(0)], _TupleInt_1[i]
            ),
        )
    ],
)
